In [17]:
# =============================================================================
# QUICKLOOK SINGLE DAY – IDW vs KRIGING (WITH OBS OVERLAYS)
# Date: 2025-04-01
# Variables: mros_plp_proxy, temp_air, temp_dew, temp_wet, rh
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pyproj import Transformer


# -----------------------------------------------------------------------------
# USER SETTINGS
# -----------------------------------------------------------------------------

DATA_DIR = Path("C:/Users/EmmaGolub/Desktop/MRoS_local/local_data")

DATA_DIR2 = Path(
    "C:/Users/EmmaGolub/Desktop/MRoS_local/mros-precipitation-phase-product-prototype/outputs/hourly_pipeline/hourly_data")

BASE_OUTPUT_DIR = Path(
    "C:/Users/EmmaGolub/Desktop/MRoS_local/"
    "mros-precipitation-phase-product-prototype/outputs/hourly_pipeline/maps"
)

PATHS = {
    "IDW": DATA_DIR / "IDW/hourly_predictors_1km_IDW_full.nc",
    "kriging": DATA_DIR / "kriging/hourly_predictors_1km_kriging_v1_final.nc"
}

STATION_PATH = DATA_DIR2 / "stations_hourly.parquet"
MROS_PATH    = DATA_DIR2 / "mros_hourly.parquet"

TARGET_DATE = "2025-04-01"

VARS_TO_SHOW = [
    "mros_plp_proxy",
    "temp_air",
    "temp_dew",
    "temp_wet",
    "rh"
]


# -----------------------------------------------------------------------------
# LOAD OBS DATA
# -----------------------------------------------------------------------------

print("Loading observation data...")
st_hr   = pd.read_parquet(STATION_PATH)
mros_hr = pd.read_parquet(MROS_PATH)


# -----------------------------------------------------------------------------
# QUICKLOOK FUNCTION
# -----------------------------------------------------------------------------

def create_quicklook(ds, method_name, out_dir):

    print(f"\nProcessing {method_name}...")
    print(ds["x"].values.min(), ds["x"].values.max())
    print(ds["y"].values.min(), ds["y"].values.max())

    print(ds)
    print("\nGlobal attrs:")
    print(ds.attrs)
    # -------------------------------------------------------------------------
    # MATCH TIME
    # -------------------------------------------------------------------------

    times_ds = pd.to_datetime(ds.time.values).floor("h")
    mask = times_ds.normalize() == pd.to_datetime(TARGET_DATE)

    if not mask.any():
        print(f"No timestep found for {TARGET_DATE}")
        return

    t_plot = times_ds[mask][0]
    ti = np.where(times_ds == t_plot)[0][0]

    print(f"Using timestep: {t_plot}")

    # -------------------------------------------------------------------------
    # GRID EXTENT
    # -------------------------------------------------------------------------

    xvals = ds["x"].values
    yvals = ds["y"].values

    xmin, xmax = xvals.min(), xvals.max()
    ymin, ymax = yvals.min(), yvals.max()
    extent = [xmin, xmax, ymin, ymax]

    # -------------------------------------------------------------------------
    # PROJECT OBSERVATIONS
    # -------------------------------------------------------------------------

    if "spatial_ref" not in ds:
        raise ValueError("No spatial_ref variable found in dataset.")

    crs_wkt = ds["spatial_ref"].attrs.get("crs_wkt")

    if crs_wkt is None:
        raise ValueError("spatial_ref exists but no crs_wkt found.")

    tf = Transformer.from_crs("EPSG:4326", crs_wkt, always_xy=True)

    # Filter to matching hour (tz-naive comparison)
    # Ensure obs times are tz-naive
    if st_hr["hour_utc"].dt.tz is not None:
        st_time = st_hr["hour_utc"].dt.tz_convert(None)
    else:
        st_time = st_hr["hour_utc"]

    if mros_hr["hour_utc"].dt.tz is not None:
        mros_time = mros_hr["hour_utc"].dt.tz_convert(None)
    else:
        mros_time = mros_hr["hour_utc"]

    st_t = st_hr[st_time.dt.floor("h") == t_plot]
    mros_t = mros_hr[mros_time.dt.floor("h") == t_plot]

    st_x, st_y = np.array([]), np.array([])
    if len(st_t):
        sx, sy = tf.transform(st_t["lon"].values, st_t["lat"].values)
        sx, sy = np.asarray(sx), np.asarray(sy)
        mask = (sx >= xmin) & (sx <= xmax) & (sy >= ymin) & (sy <= ymax)
        st_x, st_y = sx[mask], sy[mask]

    mo_x, mo_y = np.array([]), np.array([])
    if len(mros_t):
        mx, my = tf.transform(mros_t["lon"].values, mros_t["lat"].values)
        mx, my = np.asarray(mx), np.asarray(my)
        mask = (mx >= xmin) & (mx <= xmax) & (my >= ymin) & (my <= ymax)
        mo_x, mo_y = mx[mask], my[mask]

    print(f"Overlay → Stations: {len(st_x)}, MRoS: {len(mo_x)}")
    print("Projected station X range:", sx.min(), sx.max())
    print("Grid X range:", xmin, xmax)

    # -------------------------------------------------------------------------
    # VARIABLES
    # -------------------------------------------------------------------------

    keep = [v for v in VARS_TO_SHOW if v in ds.data_vars]
    if not keep:
        print("No matching variables in dataset.")
        return

    # -------------------------------------------------------------------------
    # FIGURE
    # -------------------------------------------------------------------------

    ncols = 3
    nrows = int(np.ceil(len(keep) / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.5 * ncols, 3.8 * nrows),
        squeeze=False
    )

    fig.suptitle(f"{method_name} — {t_plot:%Y-%m-%d %H:%M UTC}")

    for i, var in enumerate(keep):

        ax = axes[i // ncols, i % ncols]
        arr = ds[var].isel(time=ti).values

        if var in ("mros_plp_proxy", "rh"):
            im = ax.imshow(arr, origin="upper", extent=extent,
                           aspect="equal", vmin=0, vmax=100)
        else:
            im = ax.imshow(arr, origin="upper", extent=extent,
                           aspect="equal")

        ax.set_title(var)
        ax.set_xlabel("Easting (km)")
        ax.set_ylabel("Northing (km)")
        ax.ticklabel_format(style="plain")

        ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
        ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])

        # Overlays
        if st_x.size:
            ax.scatter(st_x, st_y, s=15,
                       c="white", edgecolor="k",
                       linewidths=0.5, marker="o",
                       zorder=3, label="Stations")

        if mo_x.size:
            ax.scatter(mo_x, mo_y, s=25,
                       c="red", edgecolor="k",
                       linewidths=0.6, marker="^",
                       zorder=3, label="MRoS")

        ax.legend(loc="upper right", fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    # Turn off unused axes
    for j in range(len(keep), nrows * ncols):
        axes[j // ncols, j % ncols].axis("off")

    # -------------------------------------------------------------------------
    # SAVE
    # -------------------------------------------------------------------------

    out_dir.mkdir(parents=True, exist_ok=True)
    out_png = out_dir / f"{method_name}_quicklook_{TARGET_DATE}.png"
    fig.subplots_adjust(
        left=0.05,
        right=0.97,
        top=0.92,
        bottom=0.06,
        wspace=0.01,   # horizontal spacing
        hspace=0.25    # vertical spacing
    )
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

    print(f"Saved → {out_png}")


# -----------------------------------------------------------------------------
# RUN BOTH METHODS
# -----------------------------------------------------------------------------

print("\nStarting quicklook generation...")

for method_name, path in PATHS.items():

    print(f"\nLoading {method_name}: {path}")

    if not path.exists():
        print("File not found. Skipping.")
        continue

    ds = xr.open_dataset(path)

    method_output_dir = BASE_OUTPUT_DIR / f"quicklooks_{method_name}"

    create_quicklook(ds, method_name, method_output_dir)

    ds.close()

print("\nAll quicklooks complete.")

Loading observation data...

Starting quicklook generation...

Loading IDW: C:\Users\EmmaGolub\Desktop\MRoS_local\local_data\IDW\hourly_predictors_1km_IDW_full.nc

Processing IDW...
162630.9182594687 303630.9182594687
4175926.168783374 4425926.168783374
<xarray.Dataset> Size: 5GB
Dimensions:         (time: 5832, y: 251, x: 142)
Coordinates:
  * y               (y) float64 2kB 4.426e+06 4.425e+06 ... 4.177e+06 4.176e+06
  * x               (x) float64 1kB 1.626e+05 1.636e+05 ... 3.026e+05 3.036e+05
  * time            (time) datetime64[ns] 47kB 2024-10-01 ... 2025-05-31T23:0...
Data variables:
    temp_air        (time, y, x) float32 831MB ...
    temp_dew        (time, y, x) float32 831MB ...
    temp_wet        (time, y, x) float32 831MB ...
    rh              (time, y, x) float32 831MB ...
    mros_plp_proxy  (time, y, x) float32 831MB ...
    plp             (time, y, x) float32 831MB ...
    elev            (y, x) float32 143kB ...
    spatial_ref     int64 8B ...
Attributes:
    

C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37120\1730395741.py:186: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37120\1730395741.py:187: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37120\1730395741.py:186: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37120\1730395741.py:187: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
 

Saved → C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\quicklooks_IDW\IDW_quicklook_2025-04-01.png

Loading kriging: C:\Users\EmmaGolub\Desktop\MRoS_local\local_data\kriging\hourly_predictors_1km_kriging_v1_final.nc

Processing kriging...
162630.9182594687 303630.9182594687
4175926.168783374 4425926.168783374
<xarray.Dataset> Size: 5GB
Dimensions:         (time: 5832, y: 251, x: 142)
Coordinates:
  * y               (y) float64 2kB 4.426e+06 4.425e+06 ... 4.177e+06 4.176e+06
  * x               (x) float64 1kB 1.626e+05 1.636e+05 ... 3.026e+05 3.036e+05
  * time            (time) datetime64[ns] 47kB 2024-10-01 ... 2025-05-31T23:0...
Data variables:
    temp_air        (time, y, x) float32 831MB ...
    temp_dew        (time, y, x) float32 831MB ...
    temp_wet        (time, y, x) float32 831MB ...
    rh              (time, y, x) float32 831MB ...
    mros_plp_proxy  (time, y, x) float32 831MB ...
    plp             (time

C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37120\1730395741.py:186: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37120\1730395741.py:187: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37120\1730395741.py:186: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37120\1730395741.py:187: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
 

Saved → C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\quicklooks_kriging\kriging_quicklook_2025-04-01.png

All quicklooks complete.
